# SuAVE Notebook Dispatcher

**Binder**: survey parameters were loaded automatically — just run all cells.  
**Colab**: paste the session token SuAVE copied to your clipboard into `SUAVE_TOKEN`,
enter your SuAVE server URL in `SUAVE_HOST`, then run all cells.

In [ ]:
# ── Cell 1: Load SuAVE parameters ─────────────────────────────────────────
import sys
sys.path.insert(0, 'helpers')
import suave_utils as su
import ipywidgets as widgets
from IPython.display import display, HTML

# ---- Colab only: paste values from the SuAVE launch dialog ---------------
SUAVE_TOKEN = ''   # @param {type:"string"}
SUAVE_HOST  = ''   # @param {type:"string"}  e.g. https://suave.sdsc.edu
# --------------------------------------------------------------------------

params = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)

if params:
    su.show_params(params)
else:
    display(HTML(
        '<p style="color:#e74c3c;font-weight:bold">'
        'Parameters not found. On Colab, enter SUAVE_TOKEN and SUAVE_HOST above '
        'and re-run this cell.'
        '</p>'
    ))

In [ ]:
# ── Cell 2: Choose an operation ────────────────────────────────────────────
#
# Menu format: (Display label, notebook path from repo root, availability)
#   'any'    — always shown
#   'image'  — only when the survey has images (dzc field is set)
#   'netvis' — only when the survey has network visualization data

MENU = [
    # ── Data / statistics ──────────────────────────────────────────────────
    ('Descriptive Statistics',              'operations/stats/DescriptiveStats.ipynb',              'any'),
    ('Generate Contingency Tables',         'operations/stats/Generate_Contingency_Tables.ipynb',   'any'),
    ('Generate Factor Contributions',       'operations/stats/Generate_Factor_Contributions.ipynb', 'any'),
    ('Arithmetic Operations',               'operations/arithmetic/SuaveArithmetic.ipynb',          'any'),
    ('Spatial Statistics',                  'operations/spatialstats/SpatialStats.ipynb',           'any'),
    ('Generate Maps',                       'operations/maps/Generate_Aggregate_Maps_Suave.ipynb',  'any'),
    ('Network Analysis',                    'operations/networks/networks.ipynb',                   'netvis'),
    ('Knowledge Graph Query',               'operations/kg/kg_query.ipynb',                        'any'),
    # ── Text / NLP ─────────────────────────────────────────────────────────
    ('Named Entity Recognition',            'operations/tagger/NER.ipynb',                         'any'),
    ('NeMo NLP',                            'operations/nemo/suave_nemo.ipynb',                    'any'),
    # ── SDG ────────────────────────────────────────────────────────────────
    ('Generate SDG Dataset',                'operations/SDG/GenerateSDGDataset.ipynb',              'any'),
    # ── Data wrangling ─────────────────────────────────────────────────────
    ('Geographic + Qualitative Imaging',    'operations/wrangling/qualgeoimage.ipynb',              'any'),
    # ── Image-based (shown only when survey has images) ────────────────────
    ('Color Statistics',                    'operations/colors/ColorStats.ipynb',                  'image'),
    ('Classify Images (Clarifai)',          'operations/classify/ImageClassify.ipynb',             'image'),
    ('Predictive Model (LeNet CNN)',        'operations/predict/PredictiveModel_v2.ipynb',         'image'),
    ('Extend Predictive Model',            'operations/predict/ExtendModel.ipynb',                'image'),
    ('SVM Predictive Model',               'operations/svm/SVMPredictiveModel.ipynb',             'image'),
    ('Extend SVM Model',                   'operations/svm/ExtendSVM.ipynb',                      'image'),
    ('Transfer Learning',                  'operations/transfer_learning/transfer_learning.ipynb','image'),
    ('HoloViz Dashboard',                  'operations/holoviz/holoviz.ipynb',                    'image'),
]

if params is None:
    raise RuntimeError('Run Cell 1 first.')

has_images = bool(params.get('dzc'))
has_netvis = False  # set True if survey has netvis data (check params['survey'] or a server call)

nb_map = {
    label: path
    for label, path, kind in MENU
    if kind == 'any'
    or (kind == 'image'  and has_images)
    or (kind == 'netvis' and has_netvis)
}

dropdown = widgets.Dropdown(
    options=list(nb_map.keys()),
    description='Operation:',
    layout=widgets.Layout(width='480px'),
    style={'description_width': '100px'},
)

btn = widgets.Button(
    description='Open Notebook',
    button_style='primary',
    icon='external-link',
    layout=widgets.Layout(width='170px'),
)

link_area = widgets.Output()

def on_click(b):
    nb_path = nb_map[dropdown.value]
    url = su.make_nb_url(nb_path)
    with link_area:
        link_area.clear_output(wait=True)
        display(HTML(
            f'<a href="{url}" target="_blank" '
            f'style="font-size:14px;font-weight:600;color:#6366f1;'
            f'text-decoration:none;padding:6px 14px;border:2px solid #6366f1;'
            f'border-radius:4px;display:inline-block;margin-top:8px">'
            f'&#9654;&nbsp; Open: {dropdown.value}'
            f'</a>'
        ))

btn.on_click(on_click)
display(widgets.VBox([dropdown, btn, link_area]))